In [ ]:
import ccxtpro
import os
import logging
import asyncio

# Configure logging
logging.basicConfig(level=logging.INFO)

# API keys and initialization
api_key = os.getenv('BINANCE_API_KEY')
api_secret = os.getenv('BINANCE_API_SECRET')

# Initialize ccxtpro Binance exchange instance
exchange = ccxtpro.binance({
    'apiKey': api_key,
    'secret': api_secret,
    'enableRateLimit': True
})
exchange.options['defaultType'] = 'future'  # Set default type to future for perp

# Parameters
ACCEPTABLE_BASIS_THRESHOLD = 50  # Basis threshold in USD
RISK_TOLERANCE = 20  # Maximum allowable basis loss in USD
CAPITAL_IN_USDT = 1000  # USD to trade
LEVERAGE = 2  # Leverage for perp position
FEE_PERCENTAGE = 0.001  # 0.1% fee
SLIPPAGE_TOLERANCE = 0.01  # 1% slippage tolerance
SL_BASIS = 20  # Stop-loss basis threshold in USD
TP_BASIS = 0   # Take-profit basis threshold in USD


# Helper functions for risk management and trading logic
def calculate_fees(spot_position_size, perp_position_size, spot_price, perp_price):
    spot_fee = spot_position_size * spot_price * FEE_PERCENTAGE
    perp_fee = perp_position_size * perp_price * FEE_PERCENTAGE
    return spot_fee + perp_fee


def check_slippage(expected_price, actual_price):
    slippage = abs(expected_price - actual_price) / expected_price
    if slippage > SLIPPAGE_TOLERANCE:
        logging.warning(f"Slippage too high: {slippage * 100:.2f}%.")
        return False
    return True


async def close_positions(spot_position_size, perp_position_size):
    try:
        if spot_position_size > 0:
            await exchange.create_market_sell_order('BTC/USDT', spot_position_size)
            logging.info(f"Closed spot position of {spot_position_size} BTC")
        if perp_position_size > 0:
            await exchange.create_market_buy_order('BTC/USDT', perp_position_size)
            logging.info(f"Closed perp position of {perp_position_size} BTC")
    except Exception as e:
        logging.error(f"Error closing positions: {e}")


# Monitor positions based on live market data
async def monitor_positions(spot_position_size, perp_position_size, initial_basis):
    while True:
        try:
            # Get the latest price data
            spot_ticker = await exchange.watch_ticker('BTC/USDT')
            perp_ticker = await exchange.watch_ticker('BTCUSDT')

            spot_price = spot_ticker['last']
            perp_price = perp_ticker['last']

            # Calculate the basis
            current_basis = perp_price - spot_price
            basis_change = current_basis - initial_basis

            logging.info(f"Spot price: {spot_price}, Perp price: {perp_price}, Basis: {current_basis}")

            # Check stop-loss or take-profit conditions
            if basis_change >= SL_BASIS:
                logging.warning("Stop-loss triggered based on basis widening.")
                await close_positions(spot_position_size, perp_position_size)
                break
            elif basis_change <= TP_BASIS:
                logging.info("Take-profit triggered based on basis narrowing.")
                await close_positions(spot_position_size, perp_position_size)
                break

            await asyncio.sleep(1)  # Sleep briefly to avoid overloading the WebSocket

        except Exception as e:
            logging.error(f"Error monitoring positions: {e}")
            break


# Main function for executing the funding rate arbitrage strategy
async def funding_rate_arbitrage(capital_in_usdt):
    try:
        # Subscribe to real-time market data streams for spot and perp
        spot_ticker = await exchange.watch_ticker('BTC/USDT')
        perp_ticker = await exchange.watch_ticker('BTCUSDT')

        spot_price = spot_ticker['last']
        perp_price = perp_ticker['last']

        # Initial basis
        initial_basis = perp_price - spot_price

        if abs(initial_basis) > ACCEPTABLE_BASIS_THRESHOLD:
            logging.warning(f"Basis {initial_basis} exceeds acceptable threshold.")
            return

        # Position sizes
        spot_position_size = capital_in_usdt / spot_price
        perp_position_size = spot_position_size * LEVERAGE

        logging.info(f"Placing orders. Spot position size: {spot_position_size}, Perp position size: {perp_position_size}")

        # Open positions
        spot_order = await exchange.create_market_buy_order('BTC/USDT', spot_position_size)
        perp_order = await exchange.create_market_sell_order('BTCUSDT', perp_position_size)

        # Check for slippage
        executed_spot_price = spot_order['average']
        executed_perp_price = perp_order['average']

        if not check_slippage(spot_price, executed_spot_price) or not check_slippage(perp_price, executed_perp_price):
            logging.warning("High slippage detected. Closing positions.")
            await close_positions(spot_position_size, perp_position_size)
            return

        # Calculate fees
        total_fees = calculate_fees(spot_position_size, perp_position_size, spot_price, perp_price)
        logging.info(f"Total fees: {total_fees} USDT")

        # Monitor positions in real-time for stop-loss and take-profit
        await monitor_positions(spot_position_size, perp_position_size, initial_basis)

    except Exception as e:
        logging.error(f"Error executing arbitrage strategy: {e}")


if __name__ == "__main__":
    capital = CAPITAL_IN_USDT
    loop = asyncio.get_event_loop()
    loop.run_until_complete(funding_rate_arbitrage(capital))
